## Understanding of Selected Paper

### Task 1.1: Core Contribution / Architecture


Step 1: Candidate Generation:

1. All possible contiguous subsequences of lengths between MINLEN and MAXLEN are extracted from every training time series in the dataset D.
2. Purpose: This creates a pool of potential shapelets (discriminative local patterns) to evaluate for class separation, focusing on local subsequences rather than full series to capture motifs like spikes or dips.
3. Reference: Section 3.1 introduces the brute-force baseline for shapelet discovery, emphasizing exhaustive generation of candidates to ensure optimality. Table 3 details the GenerateCandidates procedure: it iterates over time series lengths l from MINLEN to MAXLEN and starting positions i, appending each subsequence S = T[i:i+l] to a list (pseudocode: for each time series T in D; for l = MINLEN to MAXLEN; for i = 1 to |T|-l+1; append T[i:i+l]). This supports the step by providing the foundational pool for evaluation, with O(k m²) complexity (k series, m average length), highlighting why local focus (vs. full series) captures motifs like spikes—though expensive, it baselines optimizations in later steps.



Step 2: Distance Calculation with Early Abandon

1. For each candidate shapelet S, the minimum Euclidean distance is computed to any matching-length subsequence in each training series T, using early abandon to halt computation if a partial distance sum exceeds the current minimum.
2. Purpose: Efficiently measures how "close" a series is to the shapelet motif, enabling fast screening of candidates without full alignments.
3. Reference: Definition 5 defines SubsequenceDist(T, S) as min_{i=1 to |T|-|S|+1} Distance(T[i:i+|S|], S), typically Euclidean (sum of squared differences). Table 6 provides the SubsequenceDistanceEarlyAbandon pseudocode: it initializes min_dist to infinity, then for each starting i, computes partial sums of (T[j] - S[j-i+1])², abandoning if partial > min_dist (e.g., if partial_sum > min_dist: break). Figure 7 demos speedup: left panel shows full O(|T|) computation; right shows abandonment mid-way when partial exceeds min, yielding 2–5x faster on average. This directly enables efficient screening by reducing per-candidate cost from O(m) to constant factors, crucial for brute-force feasibility.

Step 3: Information Gain Evaluation

1. A histogram of distances is built, and the optimal split point (OSP) is found that maximizes information gain by bipartitioning classes based on distance thresholds.
2. Purpose: Quantifies how well the shapelet separates classes (e.g., low distance = class A, high = class B), selecting the most discriminative motif.
3. Reference: Definitions 6–7 define entropy I(D) = -p(A)log₂(p(A)) - p(B)log₂(p(B)) for binary classes, and Gain(sp) = I(D) - [|D₁|/|D| · I(D₁) + |D₂|/|D| · I(D₂)], where sp bipartitions via threshold. Table 5's CalculateInformationGain algo: builds distance histogram, scans adjacent bins for OSP (mean of bin edges maximizing Gain), partitions D into D₁ (dist < OSP) and D₂ (> OSP), then computes gain. This quantifies separation (e.g., high gain if low-dist cluster is mostly class A), supporting motif selection by scoring candidates' discriminative power—e.g., paper shows gains up to 0.8 on arrowheads data.



Step 4: Entropy Pruning

1. Candidates are skipped if their optimistic upper-bound gain (based on entropy) is below the best-so-far, using round-robin object ordering for better bounds.
2. Purpose: Dramatically reduces computation from O(m³) to feasible levels by eliminating unpromising shapelets early.
3. Reference: Section 3.3 explains admissible pruning: after partial distances for some objects, estimate upper-bound gain by assigning remaining objects to histogram extremes (e.g., class A to low-dist bin, B to high). Table 7's EntropyEarlyPrune pseudocode: for each class, shift unprocessed objects to optimistic bins, recompute I(D₁) and I(D₂); if max possible Gain < best_so_far, prune (e.g., if optimistic_gain_A < best and optimistic_gain_B < best: return true). Figure 11 graphs speedup on 1M-point dataset: brute-force ~days, +early abandon ~half, +pruning ~100x faster, combined ~1000x. This reduces O(m³) to practical by culling 99% of candidates early, using round-robin (process one from each class alternately) for tighter bounds.
                                                    

Step 5: Shapelet Selection and Tree Building

1. The highest-gain shapelet is selected recursively to build a decision tree, where each node splits on distance to its shapelet and threshold.
2. Purpose: Forms an interpretable classifier where traversal compares test series distances to shapelets for leaf label prediction.
3. Reference: Definition 9 defines a shapelet as subsequence S maximizing Gain(S, d^{OSP}(D, S)) over all S' in D. Section 4 describes tree induction: root finds best shapelet via discovery, splits D into D_left (dist < threshold) and D_right (>), recurses until pure leaves or depth limit. Tables 8–9: Table 8's CalculateAccuracy tests tree on holdout by recursive prediction and % correct; Table 9's Predict pseudocode: at node, compute SubsequenceDist(T, S); if < split_point, recurse left else right, return leaf label. This builds interpretable trees (e.g., 5–10 nodes deep), supporting classification by hierarchically refining local motifs for high accuracy (e.g., 80–95% on UCR datasets).

Step 6: Classification

1. For a test series, compute subsequence distances to root shapelet; if below threshold, recurse left, else right, until leaf class.
2. Purpose: Produces final class label, 3–4x faster than full-series baselines per the paper.
3. Reference: Section 4 details prediction as tree traversal using discovered shapelets. Figure 13 visualizes a 3-class tree on projectile points: top shows shapelet "dictionary" (e.g., S1: hafting motif, threshold 11.24; S2: blade, 85.47); bottom depicts nodes splitting Clovis/Avonlea/mixed via distances, with leaves labeled. This supports fast inference (3–4x vs. 1-NN, O(depth · m) time), emphasizing interpretability—e.g., "Clovis fails if dist to S1 >11.24"—and ties back to discovery for end-to-end use.


Final Summary Sentence

This paper solves the problem of interpretable time-series classification by introducing shapelets as local discriminative primitives, claiming superiority over full-series methods like 1-NN by capturing motifs for higher accuracy (e.g., 93% vs. 91% on Gun/NoGun) and 1000x faster discovery via pruning.
